# Feature Engineering + Model Selection Workflow (Project Baseline Build)

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/09_tuning_feature_engineering_project_baseline_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Engineer features with pipelines without leakage
2. Use `GridSearchCV` / `RandomizedSearchCV` for systematic tuning
3. Define a project-grade evaluation plan (metric + split/CV + baseline + reporting)
4. Produce a baseline model notebook that can be extended
5. Use Gemini to draft search grids and then simplify them

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## 💼 Why This Matters: Squeezing Out Every Percentage Point

The **State Health Department's** review board sets a bar: *"We need at least 97% recall on malignant cases before we can approve deployment."* Your current model is close but not there. Two levers remain: hyperparameter tuning (systematically searching for the best model settings) and feature engineering (creating new features that capture patterns the raw data misses).

This is the craft of applied machine learning — combining domain knowledge with systematic search to push model performance to its practical limit.

> **Today's focus:** Systematic hyperparameter tuning with GridSearchCV and RandomizedSearchCV, and engineering features to improve our breast cancer classifier.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import uniform, randint
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
print("✓ Setup complete!")

**Reading the output:**

The setup cell imports the tuning machinery we will use: `GridSearchCV` for exhaustive parameter search, `RandomizedSearchCV` for sampling-based search, and `StratifiedKFold` for the inner CV loop that evaluates each parameter combination. We also import `PolynomialFeatures` and `SelectKBest` for feature engineering, and `scipy.stats.uniform` and `randint` for defining continuous and integer parameter distributions in randomized search. The **"Setup complete!"** message confirms all libraries loaded without error.

**Why this matters:** Grid search and randomized search are the two workhorses of hyperparameter tuning in scikit-learn. Understanding when to use each -- and how they interact with pipelines and CV -- is essential for building strong models without overfitting to the validation set.

---

## 1. Load Data and Baseline

Every tuning experiment needs a **baseline** -- a simple model with default hyperparameters that establishes the performance floor. Without a baseline, you cannot tell whether elaborate feature engineering or exhaustive grid searches actually improved anything.

We continue with the breast cancer Wisconsin dataset (569 samples, 30 features) and the standard 60/20/20 split. The baseline is a Logistic Regression pipeline with `StandardScaler` and all default hyperparameters (`C=1.0`, `penalty='l2'`). Its validation accuracy will be our reference point for every experiment in this notebook.

> 💡 **Gemini Prompt:** "Load the breast cancer dataset from sklearn. Split into 60/20/20 train/val/test with stratification and seed 474. Fit a baseline StandardScaler + LogisticRegression pipeline and print the validation accuracy."
>
> **After running, verify:**
> - Train/Val/Test sizes are roughly 60%/20%/20%
> - Baseline validation accuracy is printed (above 0.95)
> - Stratified splits preserve class proportions


In [ ]:
# Load data
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

# Split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Features: {X.shape[1]}")

# Simple baseline
baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
])

baseline.fit(X_train, y_train)
baseline_score = baseline.score(X_val, y_val)

print(f"\nBaseline validation accuracy: {baseline_score:.4f}")

**Reading the output:**

The split sizes are printed as **Train: 341 | Val: 114 | Test: 114**, matching the 60/20/20 convention. The dataset has **30 features**, all numeric measurements of cell nuclei.

The baseline Logistic Regression achieves a **validation accuracy around 0.96-0.97**. This is already high because the breast cancer dataset is relatively well-separated, but there is still room for improvement -- especially if we can identify which features matter most or find non-linear relationships.

**Key takeaway:** Record this baseline number. Every subsequent experiment (feature selection, polynomial features, grid search) must be compared against it. If a complex model does not beat the baseline, the extra complexity is not justified.

---

## 2. Feature Engineering Inside Pipelines

### Safe Feature Engineering Patterns

**Rule:** All feature engineering must happen INSIDE the pipeline

**Why?**
- Prevents leakage (fit on train, transform on val/test)
- Ensures reproducibility
- Makes deployment easier

**Common feature engineering steps:**
1. Polynomial features
2. Feature interactions
3. Feature selection
4. Domain-specific transformations

> 💡 **Gemini Prompt:** "Build a pipeline that chains StandardScaler, SelectKBest (k=20), and LogisticRegression. Fit on training data, evaluate on validation, and print the improvement over baseline."
>
> **After running, verify:**
> - Pipeline has three named steps
> - Validation accuracy and improvement over baseline are displayed
> - Selected features are extracted using get_support()


In [ ]:
# Pipeline with feature engineering
fe_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest(f_classif, k=20)),  # Keep top 20 features
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
])

fe_pipeline.fit(X_train, y_train)
fe_score = fe_pipeline.score(X_val, y_val)

print("=== FEATURE ENGINEERING PIPELINE ===")
print(f"Original features: {X.shape[1]}")
print(f"Selected features: 20")
print(f"Validation accuracy: {fe_score:.4f}")
print(f"Improvement: {(fe_score - baseline_score):.4f}")

# See which features were selected
selected_mask = fe_pipeline.named_steps['feature_selection'].get_support()
selected_features = X.columns[selected_mask].tolist()
print(f"\nSelected features: {selected_features[:5]}... (showing first 5)")

**Reading the output:**

The feature engineering pipeline uses `SelectKBest(f_classif, k=20)` to keep only the 20 features with the highest ANOVA F-statistic, discarding the 10 least informative ones. The output compares:

- **Original features:** 30
- **Selected features:** 20
- **Validation accuracy:** typically close to or slightly above the baseline
- **Improvement:** the difference from baseline (may be small, positive, or even negative)

The first five selected feature names are printed to give you a sense of which measurements survived the filter. Features like `mean concave points`, `worst radius`, and `worst perimeter` tend to rank highly because they correlate strongly with the malignant/benign distinction.

**Why this matters:** Feature selection inside a pipeline prevents data leakage. The `SelectKBest` step is fitted *only* on training data, so the feature rankings do not peek at validation labels. If you performed selection outside the pipeline, you would leak information and get overly optimistic scores.

---

## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Add engineered features and re-run CV.

Try adding polynomial features (degree=2) to a subset of features.

---

> 💡 **Gemini Prompt:** "Create a pipeline with SelectKBest(k=10), PolynomialFeatures(degree=2), StandardScaler, and LogisticRegression. Compare its 5-fold CV ROC-AUC against baseline. Show the expanded feature count."
>
> **After running, verify:**
> - Polynomial pipeline uses SelectKBest BEFORE PolynomialFeatures to control explosion
> - CV scores (mean +/- std) are printed for both models
> - Feature count jumps from 10 original to ~65 polynomial features


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance


### YOUR ANALYSIS:

**Did polynomial features help?**  
[Your analysis]

**What's the cost?**  
[Feature explosion, complexity, overfitting risk]

**Would you use this in production?**  
[Justify your decision]

---

## 3. GridSearchCV - Systematic Hyperparameter Tuning

### How GridSearchCV Works

1. Define parameter grid
2. Try every combination
3. Use CV to evaluate each
4. Return best parameters

**Warning:** Grid search can be expensive!
- 3 parameters × 3 values each = 27 combinations
- 27 combinations × 5 folds = 135 model fits

> 💡 **Gemini Prompt:** "Set up GridSearchCV over a LogisticRegression pipeline, searching C=[0.01,0.1,1.0,10.0] and solver=['lbfgs','liblinear'] with 5-fold stratified CV on ROC-AUC. Print best parameters and score."
>
> **After running, verify:**
> - Total fits = 40 (4 C x 2 solvers x 5 folds)
> - Best parameters and CV ROC-AUC are printed
> - Validation score from grid_search.score() is displayed


In [ ]:
# Define pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
])

# Define parameter grid
param_grid = {
    'clf__C': [0.01, 0.1, 1.0, 10.0],
    'clf__penalty': ['l2'],  # Just L2 for speed
    'clf__solver': ['lbfgs', 'liblinear']
}

print("=== GRID SEARCH CONFIGURATION ===")
print(f"Parameter grid: {param_grid}")
n_combinations = len(param_grid['clf__C']) * len(param_grid['clf__penalty']) * len(param_grid['clf__solver'])
print(f"Total combinations: {n_combinations}")
print(f"With 5-fold CV: {n_combinations * 5} model fits")

# Run grid search
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0,
    return_train_score=True
)

print("\nRunning grid search...")
grid_search.fit(X_train, y_train)

print("\n=== GRID SEARCH RESULTS ===")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")
print(f"Validation score: {grid_search.score(X_val, y_val):.4f}")

**Reading the output:**

The grid search configuration is printed first: 4 values of `C` (0.01, 0.1, 1.0, 10.0) times 1 penalty times 2 solvers = **8 combinations**, each evaluated across 5 folds for a total of **40 model fits**.

After fitting, three key results appear:
- **Best parameters:** the combination that achieved the highest mean CV ROC-AUC. Expect `C=1.0` or `C=0.1` to win, since the breast cancer data does not require heavy regularization.
- **Best CV score:** the mean ROC-AUC of the winning combination, typically around **0.99**.
- **Validation score:** performance on the held-out validation set (which the grid search never saw). This should be close to the CV score; a large gap would indicate that the search overfit to the CV folds.

**Why this matters:** `GridSearchCV` wraps the entire search-and-evaluate loop into a single estimator. After calling `.fit()`, the object automatically refits the best parameters on the full training set, so `grid_search.predict()` uses the optimal model.

---

> 💡 **Gemini Prompt:** "Extract cv_results_ from GridSearchCV into a DataFrame, display top 5 combinations, and plot mean CV ROC-AUC vs C for each solver on a log-scale x-axis."
>
> **After running, verify:**
> - Results table shows params, mean scores, std, and rank
> - Plot has log-scale x-axis with separate lines per solver
> - Top 5 combinations sorted by rank ascending


In [ ]:
# Examine all results
results_df = pd.DataFrame(grid_search.cv_results_)
results_summary = results_df[[
    'param_clf__C', 'param_clf__solver',
    'mean_test_score', 'std_test_score',
    'mean_train_score', 'rank_test_score'
]].sort_values('rank_test_score')

print("\n=== TOP 5 PARAMETER COMBINATIONS ===")
print(results_summary.head().to_string(index=False))

# Visualize C parameter effect
plt.figure(figsize=(10, 6))
for solver in param_grid['clf__solver']:
    mask = results_df['param_clf__solver'] == solver
    plt.plot(
        results_df[mask]['param_clf__C'],
        results_df[mask]['mean_test_score'],
        marker='o', label=f'Solver: {solver}'
    )
plt.xscale('log')
plt.xlabel('C (Regularization Parameter)')
plt.ylabel('Mean CV ROC-AUC')
plt.title('Grid Search: C Parameter Effect')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Reading the output:**

The top-5 parameter combinations table shows each combination ranked by `rank_test_score`. Columns include the actual parameter values (`param_clf__C`, `param_clf__solver`), the mean and standard deviation of the test (validation) score, and the mean training score.

The line plot below visualizes how **C** (the inverse regularization strength) affects ROC-AUC for each solver. You will typically see:
- **Small C (0.01):** Strong regularization, slightly lower scores because the model is too constrained.
- **Mid-range C (0.1-1.0):** The sweet spot with highest CV scores.
- **Large C (10.0):** Weak regularization; scores may plateau or dip slightly as the model begins to overfit.

The two solver lines (lbfgs and liblinear) usually overlap closely, confirming that solver choice matters less than regularization strength for this problem.

**Key takeaway:** Examining `cv_results_` is not just good practice -- it is essential. The best parameters alone do not tell you how *sensitive* performance is to each hyperparameter. If performance is nearly flat across all C values, you have little to gain from further tuning.

---

## 4. RandomizedSearchCV - Faster Alternative

### When to Use Randomized Search

**Use RandomizedSearchCV when:**
- Parameter space is large
- Continuous parameters
- Time budget is limited
- Initial exploration phase

**Advantage:** Sample randomly instead of exhaustive search

> 💡 **Gemini Prompt:** "Create a RandomizedSearchCV over StandardScaler + RandomForestClassifier, sampling 20 combos from n_estimators(50-200), max_depth(3-20), min_samples_split(2-20), min_samples_leaf(1-10). Use ROC-AUC scoring. Print best params."
>
> **After running, verify:**
> - Parameter distributions use scipy.stats randint
> - Search runs 20 iterations (n_iter=20)
> - Best parameters and both CV and validation scores displayed


In [ ]:
# Random Forest with randomized search
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=RANDOM_SEED))
])

# Define parameter distributions
param_distributions = {
    'clf__n_estimators': randint(50, 200),
    'clf__max_depth': randint(3, 20),
    'clf__min_samples_split': randint(2, 20),
    'clf__min_samples_leaf': randint(1, 10)
}

random_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions,
    n_iter=20,  # Try 20 random combinations
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
    scoring='roc_auc',
    n_jobs=-1,
    random_state=RANDOM_SEED,
    return_train_score=True
)

print("=== RANDOMIZED SEARCH ===")
print(f"Parameter space: {param_distributions}")
print(f"Sampling 20 random combinations...")

random_search.fit(X_train, y_train)

print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.4f}")
print(f"Validation score: {random_search.score(X_val, y_val):.4f}")

**Reading the output:**

Randomized search sampled **20 random combinations** from the specified distributions: `n_estimators` between 50 and 200, `max_depth` between 3 and 20, `min_samples_split` between 2 and 20, and `min_samples_leaf` between 1 and 10. Each combination was evaluated with 5-fold stratified CV, totaling **100 model fits**.

The output reports:
- **Best parameters:** a specific set of Random Forest hyperparameters, such as `n_estimators=150, max_depth=12, min_samples_split=4, min_samples_leaf=2`.
- **Best CV score:** the mean ROC-AUC of the winning combination, likely in the **0.98-0.99** range.
- **Validation score:** performance on the held-out validation set.

Compared to GridSearchCV, randomized search explored a much larger parameter space (continuous distributions instead of fixed lists) while fitting fewer total models. The tradeoff is that it *might* miss the exact optimum, but research by Bergstra & Bengio (2012) shows that random search finds near-optimal configurations with far fewer iterations.

**Why this matters:** Use `GridSearchCV` when the parameter space is small and discrete. Use `RandomizedSearchCV` for initial exploration of large or continuous spaces, then optionally refine with a focused grid around the best region.

---

## 📝 PAUSE-AND-DO Exercise 2 (5 minutes)

**Task:** Run a small grid (2-3 params) and report best CV score.

Already done above! Now create a baseline report table:

---

> 💡 **Gemini Prompt:** "Build a baseline report DataFrame comparing three models: default LogisticRegression, grid-tuned LogisticRegression, and random-search-tuned RandomForest. Show validation ROC-AUC, parameters, and identify champion."
>
> **After running, verify:**
> - Report has Model, Val_ROC_AUC, Parameters columns
> - Three rows for the three models
> - Champion model identified by highest ROC-AUC


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance


## 5. Project Baseline Notebook Scaffold

### Required Components for Project Baseline

1. **Data Loading and Audit**
   - Load dataset
   - Check for issues
   - Document data quality

2. **Train/Val/Test Splits**
   - Proper splits with stratification
   - Lock test set away

3. **Baseline Model**
   - Simple model (mean/mode/simple classifier)
   - Establishes floor performance

4. **Improved Model**
   - Preprocessing pipeline
   - Tuned hyperparameters
   - CV evaluation

5. **Evaluation Report**
   - Multiple metrics
   - Comparison table
   - Visualizations

6. **Documentation**
   - Modeling choices explained
   - Assumptions documented
   - Next steps identified

## 6. Gemini Prompts for Tuning

### Example Prompts:

**Prompt 1: Generate Parameter Grid**
```
I'm tuning a Random Forest classifier for a binary classification task.
Generate a reasonable parameter grid for GridSearchCV including:
- n_estimators
- max_depth
- min_samples_split

Keep it small (< 20 combinations) for initial exploration.
```

**Prompt 2: Optimize Grid**
```
I ran GridSearchCV and found best params: {results}
Help me design a refined grid search around these values
to fine-tune performance.
```

**Prompt 3: Debug Search**
```
My RandomizedSearchCV is taking too long. Here's my config: {config}
Help me reduce search time while maintaining good coverage.
```

**Remember:**
- Verify Gemini's suggestions
- Start small, then expand
- Always use CV, never single split
- Document your search strategy

## 7. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Pipeline Feature Engineering**: Keep everything inside pipelines
2. **GridSearchCV**: Exhaustive search for small parameter spaces
3. **RandomizedSearchCV**: Faster exploration of large spaces
4. **Baseline Reports**: Document all models systematically
5. **Project Readiness**: Structure for reproducible modeling

### Critical Rules:

> **"All feature engineering must be in the pipeline"**

> **"Start with small grids, then refine"**

> **"Document every modeling choice"**

### Next Steps:

- Next notebook: Midterm - Business case practicum
- **Project Milestone 2 checkpoint**: Draft baseline notebook
- Apply today's patterns to your project dataset

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning with Python* - Python labs on feature engineering
- scikit-learn User Guide: [Grid search](https://scikit-learn.org/stable/modules/grid_search.html)
- scikit-learn User Guide: [Pipeline parameter tuning](https://scikit-learn.org/stable/modules/compose.html#pipeline-tuning)
- Provost, F., & Fawcett, T. (2013). *Data Science for Business* - Evaluation and business framing

---



<center>

Thank you!

</center>